# Freddie Mac Mortgage Risk Project
## Notebook 05: Challenger Model and Threshold Analysis

This notebook extends the baseline modeling stage by evaluating score thresholds, translating predicted probabilities into operational risk segments, and training a challenger model for comparison against the baseline logistic regression. The objective is to move from pure predictive benchmarking toward a more practical risk-decision framework.

## 1. Import packages and define project paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

BASE = Path("/content/drive/MyDrive/freddie_mac_risk_project")
PROCESSED = BASE / "data" / "processed"
TABLES = BASE / "outputs" / "tables"

print("Processed path exists:", PROCESSED.exists())
print("Tables path exists:", TABLES.exists())

Mounted at /content/drive
Processed path exists: True
Tables path exists: True


## 2. Load the modeling base

In [2]:
modeling_base = pd.read_parquet(PROCESSED / "modeling_base_with_target_2019_2023.parquet")

print("Modeling base shape:", modeling_base.shape)
modeling_base.head()

Modeling base shape: (250000, 34)


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_pct,num_units,occupancy_status,cltv,dti,...,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,relief_refinance_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator,vintage_year,target_12m_serious_dq
0,741,2019-03-01,N,2049-02-01,None,0,1,P,80,33,...,Other servicers,None,None,9,None,7,N,7,2019,0
1,731,2019-03-01,N,2049-02-01,13460,0,1,P,77,44,...,Other servicers,None,None,9,None,7,N,7,2019,0
2,722,2019-03-01,N,2049-02-01,None,30,1,P,95,41,...,Other servicers,None,None,9,None,7,N,N,2019,0
3,729,2019-03-01,N,2049-02-01,None,25,1,P,87,17,...,Other servicers,None,None,9,None,7,N,N,2019,0
4,773,2019-03-01,N,2049-02-01,33700,0,1,P,29,43,...,Other servicers,None,None,9,None,7,N,7,2019,0


## 3. Recreate the baseline feature set

In [3]:
baseline_features = [
    "credit_score",
    "first_time_homebuyer_flag",
    "msa",
    "mi_pct",
    "num_units",
    "occupancy_status",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "channel",
    "property_state",
    "property_type",
    "postal_code",
    "loan_purpose",
    "original_loan_term",
    "num_borrowers",
    "seller_name",
    "servicer_name",
    "program_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "mi_cancellation_indicator",
    "vintage_year"
]

target_col = "target_12m_serious_dq"

baseline_df = modeling_base[baseline_features + [target_col]].copy()

train_df = baseline_df[baseline_df["vintage_year"].isin([2019, 2020, 2021, 2022])].copy()
test_df = baseline_df[baseline_df["vintage_year"] == 2023].copy()

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (200000, 25) (200000,)
Test shape: (50000, 25) (50000,)
Train target rate: 0.011485
Test target rate: 0.00618


## 4. Rebuild the baseline logistic regression model

In [4]:
numeric_features = [
    "credit_score",
    "mi_pct",
    "num_units",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "original_loan_term",
    "num_borrowers",
    "property_valuation_method",
    "vintage_year"
]

categorical_features = [col for col in X_train.columns if col not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

logit_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

logit_model.fit(X_train, y_train)

logit_test_proba = logit_model.predict_proba(X_test)[:, 1]

print("Baseline logistic model rebuilt successfully.")

Baseline logistic model rebuilt successfully.


## 5. Evaluate threshold behavior for the logistic model

In [5]:
thresholds = [0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50]

threshold_results = []

for threshold in thresholds:
    pred_label = (logit_test_proba >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "flag_rate": pred_label.mean(),
        "precision": precision_score(y_test, pred_label, zero_division=0),
        "recall": recall_score(y_test, pred_label, zero_division=0),
        "f1": f1_score(y_test, pred_label, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_results)
threshold_results

,threshold,flag_rate,precision,recall,f1
0,0.01,0.90308,0.006555,0.957929,0.013022
1,0.02,0.85694,0.006582,0.912621,0.013069
2,0.05,0.75908,0.006850,0.841424,0.013590
3,0.10,0.62192,0.007943,0.799353,0.015730
4,0.20,0.41558,0.010299,0.692557,0.020296
5,0.30,0.26984,0.012378,0.540453,0.024201
6,0.50,0.09854,0.017455,0.278317,0.032850


## 6. Create score bands for the logistic model


In [6]:
logit_scored_test = X_test.copy()
logit_scored_test["actual_bad"] = y_test.values
logit_scored_test["score"] = logit_test_proba

logit_scored_test["risk_band"] = pd.qcut(
    logit_scored_test["score"],
    q=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"]
)

logit_band_summary = (
    logit_scored_test.groupby("risk_band")
    .agg(
        loan_count=("actual_bad", "count"),
        bad_rate=("actual_bad", "mean"),
        avg_score=("score", "mean")
    )
    .reset_index()
)

logit_band_summary

/tmp/ipykernel_164/3078380203.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  logit_scored_test.groupby("risk_band")


,risk_band,loan_count,bad_rate,avg_score
0,Very Low,10000,0.0042,0.013366
1,Low,10000,0.0023,0.071115
2,Medium,10000,0.0032,0.155784
3,High,10000,0.0078,0.279445
4,Very High,10000,0.0134,0.528817


## 7. Translate score bands into provisional policy actions


In [7]:
def map_policy_action(band):
    band = str(band)
    if band == "Very Low":
        return "Auto-approve / standard monitoring"
    elif band == "Low":
        return "Approve / standard review"
    elif band == "Medium":
        return "Manual review / request additional documentation"
    elif band == "High":
        return "Senior underwriter review / tighter exception policy"
    else:
        return "Enhanced review / possible decline"

logit_band_summary["policy_action"] = logit_band_summary["risk_band"].map(map_policy_action)
logit_band_summary

,risk_band,loan_count,bad_rate,avg_score,policy_action
0,Very Low,10000,0.0042,0.013366,Auto-approve / standard monitoring
1,Low,10000,0.0023,0.071115,Approve / standard review
2,Medium,10000,0.0032,0.155784,Manual review / request additional documentation
3,High,10000,0.0078,0.279445,Senior underwriter review / tighter exception ...
4,Very High,10000,0.0134,0.528817,Enhanced review / possible decline


## 8. Train a challenger gradient-boosting model

In [9]:
# Create fresh copies
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

# Treat property_valuation_method as categorical for the tree model
tree_numeric_features = [
    "credit_score",
    "mi_pct",
    "num_units",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "original_loan_term",
    "num_borrowers",
    "vintage_year"
]

tree_categorical_features = [col for col in X_train.columns if col not in tree_numeric_features]

print("Tree numeric features:", tree_numeric_features)
print("\nTree categorical features:", tree_categorical_features)

# Coerce numeric columns explicitly
for col in tree_numeric_features:
    X_train_tree[col] = pd.to_numeric(X_train_tree[col], errors="coerce")
    X_test_tree[col] = pd.to_numeric(X_test_tree[col], errors="coerce")

# Median imputation for numeric columns
for col in tree_numeric_features:
    median_val = X_train_tree[col].median()
    X_train_tree[col] = X_train_tree[col].fillna(median_val)
    X_test_tree[col] = X_test_tree[col].fillna(median_val)

# Fill categorical missing values
for col in tree_categorical_features:
    X_train_tree[col] = X_train_tree[col].fillna("Missing")
    X_test_tree[col] = X_test_tree[col].fillna("Missing")

# Ordinal encode categoricals
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_tree[tree_categorical_features] = ordinal_encoder.fit_transform(X_train_tree[tree_categorical_features])
X_test_tree[tree_categorical_features] = ordinal_encoder.transform(X_test_tree[tree_categorical_features])

# Train challenger model
gbm_model = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

gbm_model.fit(X_train_tree, y_train)

gbm_test_proba = gbm_model.predict_proba(X_test_tree)[:, 1]

print("Challenger model trained successfully.")

Tree numeric features: ['credit_score', 'mi_pct', 'num_units', 'cltv', 'dti', 'original_upb', 'ltv', 'interest_rate', 'original_loan_term', 'num_borrowers', 'vintage_year']

Tree categorical features: ['first_time_homebuyer_flag', 'msa', 'occupancy_status', 'channel', 'property_state', 'property_type', 'postal_code', 'loan_purpose', 'seller_name', 'servicer_name', 'program_indicator', 'property_valuation_method', 'interest_only_indicator', 'mi_cancellation_indicator']
Challenger model trained successfully.


The challenger model uses the same overall feature set as the logistic benchmark, but with a corrected treatment of coded variables for tree-based learning. In particular, `property_valuation_method` is handled as a categorical code rather than as a continuous numeric variable, which makes the preprocessing more consistent with the structure of the underlying data.

## 9. Compare baseline and challenger model performance

In [11]:
logit_test_roc_auc = roc_auc_score(y_test, logit_test_proba)
logit_test_pr_auc = average_precision_score(y_test, logit_test_proba)

gbm_test_roc_auc = roc_auc_score(y_test, gbm_test_proba)
gbm_test_pr_auc = average_precision_score(y_test, gbm_test_proba)

model_comparison = pd.DataFrame({
    "model": ["logistic_regression", "hist_gradient_boosting"],
    "test_roc_auc": [logit_test_roc_auc, gbm_test_roc_auc],
    "test_pr_auc": [logit_test_pr_auc, gbm_test_pr_auc]
})

model_comparison

,model,test_roc_auc,test_pr_auc
0,logistic_regression,0.672704,0.019128
1,hist_gradient_boosting,0.805636,0.039339


The challenger model materially outperforms the baseline logistic regression on the 2023 holdout cohort. Both ROC-AUC and PR-AUC improve substantially, indicating that the gradient-boosting model captures important nonlinear patterns and interactions that the linear benchmark does not. This suggests that the Freddie Mac origination-stage risk problem contains richer structure than can be fully represented by a simple logistic model, and it supports using the challenger as the stronger predictive benchmark going forward.

## 10. Save threshold and model comparison outputs


In [12]:
threshold_results.to_csv(TABLES / "logit_threshold_results.csv", index=False)
logit_band_summary.to_csv(TABLES / "logit_band_summary.csv", index=False)
model_comparison.to_csv(TABLES / "model_comparison_logit_vs_gbm.csv", index=False)

print(TABLES / "logit_threshold_results.csv")
print(TABLES / "logit_band_summary.csv")
print(TABLES / "model_comparison_logit_vs_gbm.csv")

/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/logit_threshold_results.csv
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/logit_band_summary.csv
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/model_comparison_logit_vs_gbm.csv


## Conclusion

The logistic regression benchmark has now been extended into a threshold-analysis framework and compared against a nonlinear challenger model. This moves the project from pure benchmark evaluation toward decision-oriented credit risk segmentation, while also testing whether a more flexible model can improve ranking performance on the 2023 holdout cohort.